# S47_02 — Document Loading and Chunking

Before embedding, documents must be loaded and split into chunks. Chunk size is one of the most important RAG hyperparameters — too small loses context, too large dilutes relevance.

## Document loading with LangChain

In [ ]:
# pip install langchain langchain-community pypdf
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    WebBaseLoader,
    DirectoryLoader,
)

# Load plain text
# loader = TextLoader('my_doc.txt')
# docs = loader.load()   # returns list of Document(page_content=..., metadata=...)

# Load PDF (one Document per page)
# loader = PyPDFLoader('paper.pdf')
# pages = loader.load()

# Load a webpage
# loader = WebBaseLoader('https://example.com')
# web_docs = loader.load()

# Load all .txt files in a directory
# loader = DirectoryLoader('./docs', glob='**/*.txt', loader_cls=TextLoader)
# all_docs = loader.load()

# Create a Document manually for demonstration
from langchain_core.documents import Document

doc = Document(
    page_content='Machine learning is a branch of artificial intelligence. It enables computers to learn from data without being explicitly programmed. Applications include image recognition, natural language processing, and recommendation systems.',
    metadata={'source': 'intro_ml.txt', 'page': 1},
)
print(doc.page_content[:100])
print('Metadata:', doc.metadata)

## Chunking strategies

In [ ]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

long_text = """Transformer models have revolutionized natural language processing.
They were introduced in the seminal paper 'Attention Is All You Need' by Vaswani et al. in 2017.
The key innovation was the self-attention mechanism, which allows each token to attend to every other token.

Unlike RNNs, transformers process sequences in parallel, making them much faster to train.
BERT uses a bidirectional encoder, while GPT uses a causal decoder.
T5 combines both with an encoder-decoder architecture for sequence-to-sequence tasks.

Modern LLMs like GPT-4, Claude, and Gemini are based on the decoder-only transformer architecture.
They are pre-trained on trillions of tokens and then instruction-tuned with RLHF."""

# Strategy 1: Fixed character chunks — simple but may split mid-sentence
char_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separator='\n',
)
char_chunks = char_splitter.split_text(long_text)
print(f'Character chunks: {len(char_chunks)}')
for i, chunk in enumerate(char_chunks):
    print(f'  [{i}] {chunk[:80]}...')

In [ ]:
# Strategy 2: Recursive splitter — tries \n\n, then \n, then space, then char
# This is the default LangChain strategy and works well for most text
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
)
recursive_chunks = recursive_splitter.split_text(long_text)
print(f'Recursive chunks: {len(recursive_chunks)}')
for i, chunk in enumerate(recursive_chunks):
    print(f'  [{i}] len={len(chunk)}: {chunk[:80]}...')

In [ ]:
# Strategy 3: Token-based splitting — respects model context limits
token_splitter = TokenTextSplitter(
    chunk_size=60,    # tokens, not characters
    chunk_overlap=10,
)
token_chunks = token_splitter.split_text(long_text)
print(f'Token chunks: {len(token_chunks)}')

# Strategy 4: Semantic chunking — split at topic boundaries (requires embeddings)
# from langchain_experimental.text_splitter import SemanticChunker
# from langchain_openai import OpenAIEmbeddings
# semantic_splitter = SemanticChunker(OpenAIEmbeddings())
# semantic_chunks = semantic_splitter.split_text(long_text)

## Chunking hyperparameters

In [ ]:
# Chunk size vs overlap tradeoffs
configs = [
    (128, 0),
    (256, 20),
    (512, 50),
    (1024, 100),
]

print(f'{'Config':20} {'Chunks':>8} {'Avg len':>10} {'Overlap %':>10}')
print('-' * 50)
for size, overlap in configs:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_text(long_text)
    avg_len = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    overlap_pct = overlap / size * 100
    print(f'size={size}, overlap={overlap:3d}   {len(chunks):>8}   {avg_len:>10.0f}   {overlap_pct:>9.0f}%')

## Document-aware chunking — preserve metadata

In [ ]:
# split_documents preserves metadata from each Document
docs = [
    Document(page_content=long_text, metadata={'source': 'transformers.txt', 'author': 'Vaswani et al.'}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
split_docs = splitter.split_documents(docs)

for i, chunk in enumerate(split_docs):
    print(f'Chunk {i}: [{chunk.metadata}]')
    print(f'  {chunk.page_content[:80]}...')
    print()

## Chunking guidelines

| Document type | Recommended strategy | Typical chunk size |
|--------------|---------------------|--------------------|
| General text | RecursiveCharacterTextSplitter | 256–512 chars |
| Code | `Language` text splitter | Per function/class |
| PDFs/reports | Per-page + recursive | 512–1024 chars |
| Q&A pairs | No chunking needed | One pair = one chunk |
| Long documents | Semantic chunker | Variable |

**Overlap** of 10–15% prevents answers that span chunk boundaries from being missed.

Next: [S47_03_embeddings_vector_databases.ipynb](./S47_03_embeddings_vector_databases.ipynb)